In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PRJ_DIR = Path().cwd()
print(PRJ_DIR)

/Users/yukihata/Desktop/quants/notebook/GEP_Quality-Growth


In [2]:
from docutils.parsers.rst.directives import encoding

json_files = list(PRJ_DIR.glob("*.json"))
dfs = []
for json_file in json_files:
    df = pd.read_json(json_file).melt(id_vars="Date")
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df = pd.pivot(df, index="Date", columns="variable", values="value")
df.index = pd.to_datetime(df.index)
display(df)

df.to_parquet(PRJ_DIR / "MSCI ACWI_FTW-LS-cum.parquet")

variable,FTW MXWD Index 1Y Fwd EPS Growth (FY) % Long-Short (High-Low) Total Return,FTW MXWD Index Communications 1Y Fwd EPS Growth (FY) % Long-Short (High-Low) Total Return,FTW MXWD Index Communications Dividend Yield Long-Short (High-Low) Total Return,FTW MXWD Index Communications Growth Long-Short (High-Low) Total Return,FTW MXWD Index Communications Low Volatility Long-Short (High-Low) Total Return,FTW MXWD Index Communications Market Capitalization Long-Short (High-Low) Total Return,FTW MXWD Index Communications Momentum Long-Short (High-Low) Total Return,FTW MXWD Index Communications Qtly EPS Acceleration Long-Short (High-Low) Total Return,FTW MXWD Index Communications Quality Long-Short (High-Low) Total Return,FTW MXWD Index Communications Size Long-Short (High-Low) Total Return,...,FTW MXWD Index Utilities Dividend Yield Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Growth Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Low Volatility Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Market Capitalization Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Momentum Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Qtly EPS Acceleration Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Quality Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Size Long-Short (High-Low) Total Return,FTW MXWD Index Utilities Value Long-Short (High-Low) Total Return,FTW MXWD Index Value Long-Short (High-Low) Total Return
Date,,,,,,,,,,,,,,,,,,,,,
2007-01-01,0.00,NaN,NaN,NaN,NaN,0.00,0.00,NaN,NaN,NaN,...,NaN,NaN,NaN,0.00,NaN,NaN,0.00,0.00,NaN,0.00
2007-01-02,-0.09,0.00,0.00,0.00,0.00,0.13,-0.07,NaN,0.00,0.00,...,0.00,0.00,0.00,0.54,0.00,NaN,0.78,0.03,0.00,-0.12
2007-01-03,-0.47,0.03,-0.25,0.06,0.29,0.62,-0.14,NaN,-0.59,0.61,...,0.85,0.24,-0.33,0.13,-0.01,NaN,-0.28,-0.57,1.59,-0.03
2007-01-04,-1.01,-0.25,-0.67,1.62,0.54,1.37,-0.41,NaN,-0.89,1.18,...,1.14,-0.86,1.35,-0.49,-0.40,NaN,-0.52,-0.16,3.21,0.18
2007-01-05,-0.91,-0.03,-1.11,2.59,1.12,1.57,-1.05,NaN,-1.10,1.34,...,1.34,-0.56,1.65,-2.63,-2.87,NaN,-0.89,-0.89,4.41,0.33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-05-27,38.99,134.73,-26.26,117.27,9.78,108.39,125.18,4.77,-24.37,84.95,...,93.28,33.00,28.27,-124.79,-0.68,25.01,-29.13,-137.24,79.14,67.61
2026-05-28,39.65,135.02,-25.78,116.60,8.76,109.09,127.44,5.76,-25.70,84.56,...,93.03,33.74,27.86,-125.64,-1.20,24.91,-29.07,-135.25,79.46,67.00
2026-05-29,40.17,135.30,-25.21,115.61,10.48,108.58,126.16,5.82,-23.25,86.39,...,94.68,34.36,29.23,-126.37,-3.48,25.00,-29.44,-133.66,81.57,68.28


## Quality factor long short performance by sector


In [ ]:
target_factors = [s for s in df.columns if "Quality" in s]

df_plot = df[target_factors].copy()
df_plot.columns = [
    s.replace("FTW MXWD Index ", "").replace(
        " Quality Long-Short (High-Low) Total Return", ""
    )
    for s in df_plot.columns
]
df_plot.rename(
    columns={"Quality Long-Short (High-Low) Total Return": "All Sectors"}, inplace=True
)
display(df_plot.head())

,Communications,Consumer Discretionary,Consumer Staples,Energy,Financials,Health Care,Industrials,Materials,All Sectors,Real Estate,Sector Neutralized,Technology,Utilities
Date,,,,,,,,,,,,,
2007-01-01,NaN,NaN,0.00,0.00,0.00,NaN,0.00,0.00,0.00,NaN,0.00,NaN,0.00
2007-01-02,0.00,0.00,0.62,-0.73,-0.18,0.00,-0.08,-0.09,-0.02,0.00,-0.01,0.00,0.78
2007-01-03,-0.59,0.26,0.37,-3.27,0.05,-0.12,0.21,-0.63,-0.25,-0.76,-0.30,-0.34,-0.28
2007-01-04,-0.89,0.29,0.10,-4.91,-0.47,0.38,-0.25,-0.26,-0.41,0.14,-0.47,-0.23,-0.52
2007-01-05,-1.10,0.23,0.07,-3.84,-0.54,0.24,-0.43,-0.31,-0.33,1.21,-0.42,-0.09,-0.89


In [13]:
fig = go.Figure()
for col in df_plot.columns:
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot[col], mode="lines", name=col))

fig.update_layout(
    title="MSCI ACWI Quality Factor Long-Short",
    template="plotly_dark",
    xaxis_title="Date",
    yaxis_title="Total Return(%)",
    yaxis=dict(side="right"),
    margin=dict(l=50, r=80, t=50, b=30),
    legend=dict(orientation="h", yanchor="top", y=-0.15, xanchor="center", x=0.5),
)


fig.show()